[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umangbagadi03/build-your-own-coding-agent/blob/main/notebooks/01_first_api_call.ipynb)

# 01. Your First LLM API Call

**Part 1: The Loop** · about 15 minutes

Every AI product, including the coding agents you use every day, talks to a model the same way: it sends an HTTP POST and reads JSON back. Before we build an agent, we'll look at that request with nothing in the way. No SDK and no framework, just `requests`.

**By the end you will know:**

- what a chat completion request and response look like
- why a model can't remember anything, and how chat apps fake it
- what a system prompt does
- why tokens are the meter that every design decision in this course comes back to

## Setup

The next cell connects to a model. **With no key it uses offline mode**: a small local server that speaks the real wire format but follows a fixed script. Your code stays the same either way, so you can read this first and plug in a real model later. Real answers will differ from the saved outputs below.

| To use | Set |
|---|---|
| Offline script (default) | nothing |
| Groq (has a free tier) | `GROQ_API_KEY` as an environment variable or Colab secret |
| Ollama on your machine | `LLM_PROVIDER=ollama` |
| Any OpenAI-compatible API | `LLM_PROVIDER=custom`, `LLM_BASE_URL`, `LLM_MODEL`, and optionally `LLM_API_KEY` |

In [1]:
import pathlib, urllib.request

if not pathlib.Path("llm_setup.py").exists():  # Colab opens one notebook at a time, so fetch the helper
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/umangbagadi03/build-your-own-coding-agent/main/notebooks/llm_setup.py", "llm_setup.py"
    )

from llm_setup import connect

llm = connect()

OFFLINE MODE: a scripted local server, not a real model.
The HTTP calls are real. The answers are canned for these lessons.
Token counts are rough estimates and timings are simulated.
Set GROQ_API_KEY (free tier) or LLM_PROVIDER=ollama to use a real model.


## 1. The smallest possible request

Three things go in: **where** to send it, **who** you are, and **what** you want to say.

In [2]:
import json

import requests

url = f"{llm.base_url}/chat/completions"

headers = {"Content-Type": "application/json"}
if llm.api_key:
    headers["Authorization"] = f"Bearer {llm.api_key}"

payload = {
    "model": llm.model,
    "messages": [
        {"role": "user", "content": "In one sentence, what is a large language model?"}
    ],
}

response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
response.raise_for_status()

data = response.json()
print(json.dumps(data, indent=2))

{
  "id": "offline-response",
  "object": "chat.completion",
  "model": "offline-script",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "A large language model is a neural network trained on huge amounts of text to predict what comes next, which lets it generate useful language."
      },
      "finish_reason": "stop"
    }
  ],
  "usage": {
    "prompt_tokens": 21,
    "completion_tokens": 37,
    "total_tokens": 58
  }
}


That's the whole thing. Three fields in the response matter most:

| Field | Meaning |
|---|---|
| `choices[0].message` | What the model said. Note that it has the same shape as the message you sent. |
| `choices[0].finish_reason` | Why it stopped. `stop` means it finished. `tool_calls` will become very important in notebook 03. |
| `usage` | How many tokens you were charged for. |

Let's pull out those pieces.

In [3]:
choice = data["choices"][0]

print("Answer:       ", choice["message"]["content"])
print("Finish reason:", choice["finish_reason"])
print("Usage:        ", data["usage"])

Answer:        A large language model is a neural network trained on huge amounts of text to predict what comes next, which lets it generate useful language.
Finish reason: stop
Usage:         {'prompt_tokens': 21, 'completion_tokens': 37, 'total_tokens': 58}


## 2. Wrap it in a helper

We'll make this call constantly, so let's give it a name. Notice how little there is to it.

In [4]:
def chat(messages, **options):
    """Send messages to the model and return the parsed JSON response."""
    payload = {"model": llm.model, "messages": messages, **options}
    response = requests.post(url, headers=headers, json=payload, timeout=llm.timeout)
    response.raise_for_status()
    return response.json()


def text_of(data):
    """Pull the assistant's text out of a response."""
    return data["choices"][0]["message"]["content"]


print(text_of(chat([{"role": "user", "content": "In one sentence, what is a large language model?"}])))

A large language model is a neural network trained on huge amounts of text to predict what comes next, which lets it generate useful language.


## 3. The model has no memory

This is the most important idea in the notebook. Watch what happens when we introduce ourselves and then ask a follow-up in a separate call.

In [5]:
first = chat([{"role": "user", "content": "Hi! My name is Asha."}])
print("Reply 1:", text_of(first))

second = chat([{"role": "user", "content": "What is my name?"}])
print("Reply 2:", text_of(second))

Reply 1: Nice to meet you, Asha!


Reply 2: I don't know your name. I can't see any earlier messages.


The second call knows nothing about the first. Every request is **stateless**: the server handles it and moves on.

So how does ChatGPT remember what you said? It doesn't. **The app resends the whole conversation with every request.** Let's do exactly that.

In [6]:
conversation = [{"role": "user", "content": "Hi! My name is Asha."}]

reply = chat(conversation)
conversation.append(reply["choices"][0]["message"])  # keep what the model said

conversation.append({"role": "user", "content": "What is my name?"})  # then add the follow-up
reply = chat(conversation)

print("Reply:", text_of(reply))
print()
print("What we sent on the second call:")
print(json.dumps(conversation, indent=2))

Reply: Your name is Asha.

What we sent on the second call:
[
  {
    "role": "user",
    "content": "Hi! My name is Asha."
  },
  {
    "role": "assistant",
    "content": "Nice to meet you, Asha!"
  },
  {
    "role": "user",
    "content": "What is my name?"
  }
]


In [7]:
# Checkpoint: the offline script is deterministic, so we can check its exact behaviour.
if llm.offline:
    assert "don't know" in text_of(second), "A separate call should have no memory."
    assert "Asha" in text_of(reply), "Resending the history should restore the context."
print("Checkpoint passed.")

Checkpoint passed.


> **Why this matters for agents:** an agent's entire memory is this one list. Everything it has read, tried and learned must fit in it, and the whole list gets resent on every step. Part 3 of this course is about what to do when it gets too big.

## 4. System prompts

A message with the `system` role sets the rules for the conversation. A coding agent's personality, its rules and its list of dos and don'ts all live here.

In [8]:
messages = [
    {"role": "system", "content": "You are a pirate. Answer in one short sentence."},
    {"role": "user", "content": "What is an API?"},
]

print(text_of(chat(messages)))

Arr, an API be a contract between programs, matey!


There's no special machinery behind this. A system prompt is text in the same list, and the model has been trained to weight it heavily. It also means a system prompt is a strong suggestion rather than a security boundary, which will matter when our agent starts running tools.

## 5. Tokens are the meter

Models don't read characters. They read **tokens**, which are chunks of text of roughly 3 to 4 English characters each. You pay for tokens going in and tokens coming out, and every model has a limit on how many it can see at once (its *context window*).

Since the history gets resent each time, input tokens climb on every turn. Let's watch that happen.

In [9]:
# Example prices only, to show the arithmetic. Real prices vary a lot by model and change often.
EXAMPLE_INPUT_PRICE = 1.00   # dollars per million tokens
EXAMPLE_OUTPUT_PRICE = 4.00  # dollars per million tokens

conversation = [{"role": "system", "content": "You are a concise coding mentor."}]
total_cost = 0.0

print("turn | input tokens | output tokens | example cost")
print("-----+--------------+---------------+-------------")
for turn in range(1, 6):
    conversation.append({"role": "user", "content": "Give me one more coding tip."})
    reply = chat(conversation)
    conversation.append(reply["choices"][0]["message"])

    usage = reply["usage"]
    cost = (
        usage["prompt_tokens"] * EXAMPLE_INPUT_PRICE
        + usage["completion_tokens"] * EXAMPLE_OUTPUT_PRICE
    ) / 1_000_000
    total_cost += cost
    print(f"{turn:>4} | {usage['prompt_tokens']:>12} | {usage['completion_tokens']:>13} | ${cost:.6f}")

print(f"\nTotal: ${total_cost:.6f}")

turn | input tokens | output tokens | example cost
-----+--------------+---------------+-------------


   1 |           33 |            18 | $0.000105


   2 |           74 |            18 | $0.000146


   3 |          115 |            18 | $0.000187


   4 |          155 |            18 | $0.000227


   5 |          196 |            18 | $0.000268

Total: $0.000933


The input column grows every turn even though each question is the same size. Keep this table in mind: an agent is a conversation with itself that can run for dozens of turns, so this growth is where most of its cost comes from.

## 6. When a request fails

APIs fail with an HTTP status code and usually a helpful JSON body. Let's send a bad request on purpose. The exact wording varies by provider.

In [10]:
bad = requests.post(url, headers=headers, json={"model": llm.model, "messages": []}, timeout=llm.timeout)

print("Status:", bad.status_code)
print("Body:  ", bad.text[:300])

Status: 400
Body:   {"error": {"message": "'messages' is required and can't be empty.", "type": "invalid_request_error"}}


In [11]:
# Checkpoint: an empty conversation must be rejected as a client error, not a server crash.
assert 400 <= bad.status_code < 500, f"Expected a 4xx error, got {bad.status_code}"
print("Checkpoint passed.")

Checkpoint passed.


A rough guide for later:

| Status | Usually means | What to do |
|---|---|---|
| 400 | The request is malformed | Fix your code. Retrying won't help. |
| 401 / 403 | Bad or missing key | Fix your credentials. |
| 429 | Rate limited | Wait and retry. |
| 5xx | Provider problem | Wait and retry. |

We'll add real retry logic in Part 4.

## Recap

- A model call is an HTTP POST with a list of messages.
- The model is stateless, so **the message list is the memory**.
- System prompts are just messages that the model weights heavily.
- Input tokens grow every turn because the history is resent.

## Exercises

1. Add a third turn to the Asha conversation that asks the model to spell your name backwards. What has to be in the list for it to work?
2. Delete the `system` message from section 4. How does the answer change on a real model?
3. Write `forget_oldest(conversation, keep)`, which drops the oldest messages but keeps the system prompt. What goes wrong when the model loses an early message? You've just met the core problem of Part 3.

**Next:** [02. Streaming](02_streaming.ipynb), or how to show words as they are generated.